In [1]:
# =====================================================================
# Identify meta-learner for each naive stacking variant
# =====================================================================
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import clone
from sklearn.metrics import f1_score, mean_absolute_error
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
import lightgbm as lgb
from scipy.special import expit

# ---- Config ----
TRAINED_ROOT = Path("./trained")
DATA_ROOT = Path("./prepareddata")
TARGET_COL = "Fraud"
RANDOM_STATE = 42
THRESHOLD = 0.5

# ---- Meta-learner pool (same as training.ipynb) ----
def get_meta_learners(random_state=42):
    return {
        "LogisticRegression": LogisticRegression(
            C=1.0, max_iter=1000, random_state=random_state
        ),
        "RidgeClassifier": RidgeClassifier(
            alpha=1.0, random_state=random_state
        ),
        "RandomForest": RandomForestClassifier(
            n_estimators=200, max_depth=4, min_samples_leaf=10,
            random_state=random_state, n_jobs=-1
        ),
        "GaussianNB": GaussianNB(),
        "MLP": MLPClassifier(
            hidden_layer_sizes=(64, 32), activation='relu',
            solver='adam', alpha=0.0001, max_iter=500,
            early_stopping=True, validation_fraction=0.1,
            n_iter_no_change=10, random_state=random_state
        ),
        "LinearSVC": LinearSVC(
            C=1.0, max_iter=2000, random_state=random_state
        ),
        "XGBoost": XGBClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05,
            subsample=0.8, colsample_bytree=0.8,
            eval_metric="auc", random_state=random_state,
            verbosity=0, use_label_encoder=False
        ),
        "LightGBM": lgb.LGBMClassifier(
            n_estimators=200, learning_rate=0.05,
            num_leaves=31, subsample=0.8, colsample_bytree=0.8,
            random_state=random_state, verbose=-1, n_jobs=-1
        ),
    }

def predict_proba(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        return expit(model.decision_function(X))
    else:
        return model.predict(X)

# ---- Main identification loop ----
records = []
meta_learners = get_meta_learners(RANDOM_STATE)

for dataset_dir in sorted(TRAINED_ROOT.iterdir()):
    if not dataset_dir.is_dir():
        continue
    ds = dataset_dir.name
    print(f"\n[dataset] {ds}")

    for variant_dir in sorted(dataset_dir.iterdir()):
        if not variant_dir.is_dir():
            continue
        variant = variant_dir.name

        val_file = variant_dir / "val_meta_features.csv"
        test_file = variant_dir / "test_meta_features.csv"
        pred_file = variant_dir / "test_predictions.csv"

        if not (val_file.exists() and test_file.exists() and pred_file.exists()):
            continue

        # Load data
        X_val = pd.read_csv(val_file)
        X_test = pd.read_csv(test_file)
        y_val = pd.read_csv(DATA_ROOT / f"{ds}_val.csv")[TARGET_COL].values
        pred_df = pd.read_csv(pred_file)

        # Use saved stack_proba if available, otherwise use y_true from pred_df
        if "stack_proba" in pred_df.columns:
            saved_proba = pred_df["stack_proba"].values
            y_test = pred_df["y_true"].values if "y_true" in pred_df.columns else None
        else:
            saved_proba = None
            y_test = None

        best_match = None
        best_mae = np.inf

        for name, meta in meta_learners.items():
            model = clone(meta)
            model.fit(X_val, y_val)
            proba = predict_proba(model, X_test)

            if saved_proba is not None:
                mae = mean_absolute_error(saved_proba, proba)
                if mae < best_mae:
                    best_mae = mae
                    best_match = name
            else:
                if y_test is not None:
                    f1 = f1_score(y_test, (proba >= THRESHOLD).astype(int), zero_division=0)
                    if best_match is None or f1 > best_f1:
                        best_f1 = f1
                        best_match = name

        if best_match:
            records.append({
                "dataset": ds,
                "variant": variant,
                "identified_meta_learner": best_match,
                "mae_vs_saved_proba": best_mae if saved_proba is not None else np.nan,
            })
            print(f"  {variant[:60]:60s} -> {best_match} (MAE={best_mae:.6f})")

# Save results
out_df = pd.DataFrame(records)
out_path = Path("./trained/identified_meta_learners.csv")
out_df.to_csv(out_path, index=False)
print(f"\n[ok] Results saved to {out_path}")
print(out_df.to_string(index=False))


[dataset] EuropeanCard
  ADASYN--ANOVA_Percentile10--EuropeanCard--20260630_063233    -> XGBoost (MAE=0.010802)
  ADASYN--ANOVA_Percentile20--EuropeanCard--20260630_063249    -> XGBoost (MAE=0.001594)
  ADASYN--ANOVA_Percentile30--EuropeanCard--20260630_063318    -> LightGBM (MAE=0.000608)
  ADASYN--ANOVA_Percentile50--EuropeanCard--20260630_063410    -> LightGBM (MAE=0.000427)
  ADASYN--ANOVA_k10--EuropeanCard--20260630_060816             -> LightGBM (MAE=0.000608)
  ADASYN--ANOVA_k15--EuropeanCard--20260630_061001             -> XGBoost (MAE=0.000555)
  ADASYN--ANOVA_k20--EuropeanCard--20260630_061358             -> XGBoost (MAE=0.000556)
  ADASYN--ANOVA_k30--EuropeanCard--20260630_061921             -> LightGBM (MAE=0.000123)
  ADASYN--ANOVA_k5--EuropeanCard--20260630_060725              -> XGBoost (MAE=0.006169)
  ADASYN--ANOVA_kall--EuropeanCard--20260630_062554            -> XGBoost (MAE=0.000416)
  ADASYN--MI_k10--EuropeanCard--20260630_060916                -> XGBoost (MAE=0.0

In [2]:
trained_root = Path("./trained")
results = []

for dataset_dir in trained_root.iterdir():
    if not dataset_dir.is_dir():
        continue
    ds = dataset_dir.name
    for variant_dir in dataset_dir.iterdir():
        if not variant_dir.is_dir():
            continue
        variant = variant_dir.name
        pred_file = variant_dir / "test_predictions.csv"
        if not pred_file.exists():
            continue
        pred_df = pd.read_csv(pred_file)
        if "stack_proba" not in pred_df.columns or "y_true" not in pred_df.columns:
            continue
        y_true = pred_df["y_true"]
        y_pred = (pred_df["stack_proba"] >= 0.5).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        results.append({"dataset": ds, "variant": variant, "f1": f1})

f1_df = pd.DataFrame(results)

# Load identified meta-learners from previous script output
# (assuming you saved it to identified_meta_learners.csv)
meta_df = pd.read_csv("./trained/identified_meta_learners.csv")

# Merge and get best per dataset
merged = f1_df.merge(meta_df, on=["dataset", "variant"], how="inner")
best_per_ds = merged.loc[merged.groupby("dataset")["f1"].idxmax()]

print(best_per_ds[["dataset", "variant", "f1", "identified_meta_learner"]])

          dataset                                            variant  \
159  EuropeanCard  EditedNearestNeighbours--ANOVA_k15--EuropeanCa...   
345      IEEE-CIS  EditedNearestNeighbours--VarianceThreshold--IE...   
437       Sparkov  BorderlineSMOTE--MI_k15--Sparkov--20260628_224236   

           f1 identified_meta_learner  
159  0.844444      LogisticRegression  
345  0.464791      LogisticRegression  
437  0.779389      LogisticRegression  
